In [ ]:
{
 "nbformat": 4,
 "nbformat_minor": 5,
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  },
  "colab": {
   "provenance": [],
   "gpuType": "A100"
  },
  "accelerator": "GPU"
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🧠 Step 2 — QLoRA Fine-Tuning\n",
    "\n",
    "**Run this notebook on Google Colab with A100 GPU.**\n",
    "\n",
    "```\n",
    "Runtime → Change runtime type → A100 GPU → Save\n",
    "```\n",
    "\n",
    "This notebook:\n",
    "1. Installs training dependencies\n",
    "2. Loads the Kannada legal QA dataset\n",
    "3. Fine-tunes IndicBERT using QLoRA\n",
    "4. Tracks training with Weights & Biases\n",
    "5. Saves the trained model to Google Drive\n",
    "\n",
    "---\n",
    "> ⚠️ Run `01_data_collection.ipynb` first before this notebook."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 1 — Check GPU"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "\n",
    "print('GPU available :', torch.cuda.is_available())\n",
    "print('GPU name      :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')\n",
    "print('GPU memory    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB' if torch.cuda.is_available() else '')\n",
    "\n",
    "if not torch.cuda.is_available():\n",
    "    print('\\n⚠️  No GPU found!')\n",
    "    print('Go to Runtime → Change runtime type → GPU → A100')\n",
    "else:\n",
    "    print('\\n✅ GPU ready for training!')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 2 — Mount Google Drive"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from google.colab import drive\n",
    "drive.mount('/content/drive')\n",
    "print('Google Drive mounted!')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 3 — Install Training Dependencies"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!pip install transformers==4.40.0 \\\n",
    "             peft==0.10.0 \\\n",
    "             bitsandbytes==0.43.0 \\\n",
    "             accelerate==0.27.0 \\\n",
    "             trl==0.8.6 \\\n",
    "             datasets==2.18.0 \\\n",
    "             wandb \\\n",
    "             sentencepiece \\\n",
    "             indic-nlp-library \\\n",
    "             loguru -q\n",
    "\n",
    "print('\\n✅ All training dependencies installed!')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 4 — Login to Weights & Biases"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import wandb\n",
    "\n",
    "# Get your API key from https://wandb.ai/settings\n",
    "wandb.login()\n",
    "\n",
    "# Set project name\n",
    "import os\n",
    "os.environ['WANDB_PROJECT'] = 'kannada-legal-ai'\n",
    "\n",
    "print('Weights & Biases ready!')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 5 — Clone Repository"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "\n",
    "GITHUB_USERNAME = 'YOUR_USERNAME'\n",
    "REPO_NAME       = 'kannada-legal-ai'\n",
    "\n",
    "if not os.path.exists(f'/content/{REPO_NAME}'):\n",
    "    !git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git\n",
    "else:\n",
    "    print('Repo exists. Pulling latest...')\n",
    "    !cd /content/{REPO_NAME} && git pull\n",
    "\n",
    "%cd /content/{REPO_NAME}\n",
    "\n",
    "# Add to Python path\n",
    "import sys\n",
    "sys.path.insert(0, f'/content/{REPO_NAME}')\n",
    "\n",
    "print(f'Working directory: {os.getcwd()}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 6 — Load Data from Google Drive (if already collected)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import shutil\n",
    "import os\n",
    "\n",
    "DRIVE_PATH = '/content/drive/MyDrive/kannada-legal-ai'\n",
    "\n",
    "# Copy data from Drive to repo\n",
    "if os.path.exists(f'{DRIVE_PATH}/data/annotated'):\n",
    "    shutil.copytree(\n",
    "        f'{DRIVE_PATH}/data/annotated',\n",
    "        'data/annotated',\n",
    "        dirs_exist_ok=True\n",
    "    )\n",
    "    shutil.copytree(\n",
    "        f'{DRIVE_PATH}/data/processed',\n",
    "        'data/processed',\n",
    "        dirs_exist_ok=True\n",
    "    )\n",
    "    print('Data loaded from Google Drive!')\n",
    "else:\n",
    "    print('No Drive data found. Run 01_data_collection.ipynb first.')\n",
    "\n",
    "# Check dataset\n",
    "import json\n",
    "for split in ['train', 'val', 'test']:\n",
    "    path = f'data/annotated/{split}/qa_pairs.jsonl'\n",
    "    if os.path.exists(path):\n",
    "        with open(path) as f:\n",
    "            count = sum(1 for _ in f)\n",
    "        print(f'  {split:10} : {count} pairs')\n",
    "    else:\n",
    "        print(f'  {split:10} : NOT FOUND')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 7 — Preview Training Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import json\n",
    "\n",
    "print('── Sample Training Pairs ──\\n')\n",
    "\n",
    "with open('data/annotated/train/qa_pairs.jsonl') as f:\n",
    "    for i, line in enumerate(f):\n",
    "        if i >= 3:\n",
    "            break\n",
    "        pair = json.loads(line)\n",
    "        print(f'[{i+1}]')\n",
    "        print(f'  Question : {pair[\"question\"]}')\n",
    "        print(f'  Answer   : {pair[\"answer\"][:100]}...')\n",
    "        print(f'  Law      : {pair.get(\"law\", \"?\")} §{pair.get(\"section\", \"?\")}')\n",
    "        print(f'  Intent   : {pair.get(\"intent\", \"?\")}\\n')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 8 — Load Model and Tokenizer"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "from transformers import (\n",
    "    AutoModelForCausalLM,\n",
    "    AutoTokenizer,\n",
    "    BitsAndBytesConfig,\n",
    ")\n",
    "\n",
    "MODEL_NAME = 'ai4bharat/indic-bert'\n",
    "\n",
    "print(f'Loading model: {MODEL_NAME}')\n",
    "\n",
    "# 4-bit quantization config (QLoRA)\n",
    "bnb_config = BitsAndBytesConfig(\n",
    "    load_in_4bit=True,\n",
    "    bnb_4bit_compute_dtype=torch.float16,\n",
    "    bnb_4bit_quant_type='nf4',\n",
    "    bnb_4bit_use_double_quant=True,\n",
    ")\n",
    "\n",
    "# Load tokenizer\n",
    "tokenizer = AutoTokenizer.from_pretrained(\n",
    "    MODEL_NAME,\n",
    "    trust_remote_code=True,\n",
    ")\n",
    "tokenizer.pad_token = tokenizer.eos_token\n",
    "\n",
    "# Load model with 4-bit quantization\n",
    "model = AutoModelForCausalLM.from_pretrained(\n",
    "    MODEL_NAME,\n",
    "    quantization_config=bnb_config,\n",
    "    device_map='auto',\n",
    "    trust_remote_code=True,\n",
    ")\n",
    "\n",
    "print(f'\\n✅ Model loaded!')\n",
    "print(f'   Parameters: {model.num_parameters():,}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 9 — Apply LoRA Adapters"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from peft import (\n",
    "    LoraConfig,\n",
    "    get_peft_model,\n",
    "    prepare_model_for_kbit_training,\n",
    "    TaskType,\n",
    ")\n",
    "\n",
    "# Prepare model for QLoRA training\n",
    "model = prepare_model_for_kbit_training(model)\n",
    "\n",
    "# LoRA configuration\n",
    "lora_config = LoraConfig(\n",
    "    r=16,\n",
    "    lora_alpha=32,\n",
    "    lora_dropout=0.05,\n",
    "    target_modules=['query', 'value'],\n",
    "    bias='none',\n",
    "    task_type=TaskType.CAUSAL_LM,\n",
    ")\n",
    "\n",
    "model = get_peft_model(model, lora_config)\n",
    "\n",
    "# Show trainable parameters\n",
    "model.print_trainable_parameters()\n",
    "\n",
    "print('\\n✅ LoRA adapters applied!')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 10 — Prepare Dataset"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from datasets import Dataset\n",
    "import json\n",
    "\n",
    "# Alpaca-style prompt template\n",
    "TEMPLATE = \"\"\"### ಸೂಚನೆ:\n",
    "ನೀವು ಒಬ್ಬ ಕನ್ನಡ ಕಾನೂನು ತಜ್ಞ. ಕೆಳಗಿನ ಪ್ರಶ್ನೆಗೆ ನಿಖರ ಉತ್ತರ ನೀಡಿ.\n",
    "\n",
    "### ಸಂದರ್ಭ:\n",
    "{context}\n",
    "\n",
    "### ಪ್ರಶ್ನೆ:\n",
    "{question}\n",
    "\n",
    "### ಉತ್ತರ:\n",
    "{answer}\"\"\"\n",
    "\n",
    "def format_pair(row):\n",
    "    return TEMPLATE.format(\n",
    "        context=row.get('context', ''),\n",
    "        question=row['question'],\n",
    "        answer=row['answer'],\n",
    "    )\n",
    "\n",
    "def load_split(split):\n",
    "    path = f'data/annotated/{split}/qa_pairs.jsonl'\n",
    "    records = []\n",
    "    with open(path) as f:\n",
    "        for line in f:\n",
    "            row = json.loads(line.strip())\n",
    "            records.append({'text': format_pair(row)})\n",
    "    return Dataset.from_list(records)\n",
    "\n",
    "train_dataset = load_split('train')\n",
    "val_dataset   = load_split('val')\n",
    "\n",
    "print(f'Train size : {len(train_dataset)}')\n",
    "print(f'Val size   : {len(val_dataset)}')\n",
    "print(f'\\nSample prompt preview:')\n",
    "print(train_dataset[0]['text'][:300])"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 11 — Configure Training Arguments"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from transformers import TrainingArguments\n",
    "\n",
    "training_args = TrainingArguments(\n",
    "    output_dir='training/checkpoints',\n",
    "\n",
    "    # Training\n",
    "    num_train_epochs=3,\n",
    "    per_device_train_batch_size=4,\n",
    "    per_device_eval_batch_size=4,\n",
    "    gradient_accumulation_steps=4,\n",
    "\n",
    "    # Learning rate\n",
    "    learning_rate=2e-4,\n",
    "    lr_scheduler_type='cosine',\n",
    "    warmup_ratio=0.03,\n",
    "\n",
    "    # Precision\n",
    "    fp16=True,\n",
    "\n",
    "    # Logging\n",
    "    logging_steps=10,\n",
    "    logging_dir='training/logs',\n",
    "    report_to='wandb',\n",
    "\n",
    "    # Saving\n",
    "    save_strategy='epoch',\n",
    "    evaluation_strategy='epoch',\n",
    "    load_best_model_at_end=True,\n",
    "    save_total_limit=2,\n",
    "\n",
    "    # Other\n",
    "    dataloader_num_workers=2,\n",
    "    remove_unused_columns=False,\n",
    ")\n",
    "\n",
    "print('Training arguments configured!')\n",
    "print(f'  Epochs     : {training_args.num_train_epochs}')\n",
    "print(f'  Batch size : {training_args.per_device_train_batch_size}')\n",
    "print(f'  LR         : {training_args.learning_rate}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 12 — Start Training"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from trl import SFTTrainer\n",
    "\n",
    "trainer = SFTTrainer(\n",
    "    model=model,\n",
    "    args=training_args,\n",
    "    train_dataset=train_dataset,\n",
    "    eval_dataset=val_dataset,\n",
    "    tokenizer=tokenizer,\n",
    "    dataset_text_field='text',\n",
    "    max_seq_length=512,\n",
    ")\n",
    "\n",
    "print('Starting training...')\n",
    "print('Track progress at: https://wandb.ai\\n')\n",
    "\n",
    "trainer.train()\n",
    "\n",
    "print('\\n✅ Training complete!')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 13 — Save Best Model"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "\n",
    "# Save locally in Colab\n",
    "SAVE_PATH = 'training/checkpoints/best_model'\n",
    "os.makedirs(SAVE_PATH, exist_ok=True)\n",
    "\n",
    "trainer.save_model(SAVE_PATH)\n",
    "tokenizer.save_pretrained(SAVE_PATH)\n",
    "\n",
    "print(f'Model saved to: {SAVE_PATH}')\n",
    "\n",
    "# List saved files\n",
    "for f in os.listdir(SAVE_PATH):\n",
    "    size = os.path.getsize(f'{SAVE_PATH}/{f}')\n",
    "    print(f'  {f} — {round(size/1e6, 1)} MB')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 14 — Save Model to Google Drive"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import shutil\n",
    "\n",
    "DRIVE_MODEL_PATH = '/content/drive/MyDrive/kannada-legal-ai/best_model'\n",
    "\n",
    "# Remove existing if present\n",
    "if os.path.exists(DRIVE_MODEL_PATH):\n",
    "    shutil.rmtree(DRIVE_MODEL_PATH)\n",
    "\n",
    "shutil.copytree(\n",
    "    'training/checkpoints/best_model',\n",
    "    DRIVE_MODEL_PATH\n",
    ")\n",
    "\n",
    "print(f'\\n✅ Model saved to Google Drive!')\n",
    "print(f'   Location: {DRIVE_MODEL_PATH}')\n",
    "print('\\nNext step: Run 03_evaluation.ipynb')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 15 — Quick Inference Test"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from transformers import pipeline\n",
    "\n",
    "# Load the saved model for inference\n",
    "pipe = pipeline(\n",
    "    'text-generation',\n",
    "    model='training/checkpoints/best_model',\n",
    "    tokenizer=tokenizer,\n",
    "    max_new_tokens=128,\n",
    "    device_map='auto',\n",
    ")\n",
    "\n",
    "# Test with sample queries\n",
    "test_prompts = [\n",
    "    '### ಪ್ರಶ್ನೆ:\\nIPC ಸೆಕ್ಷನ್ 302 ಏನು?\\n### ಉತ್ತರ:',\n",
    "    '### ಪ್ರಶ್ನೆ:\\nಕಳ್ಳತನಕ್ಕೆ ಎಷ್ಟು ಶಿಕ್ಷೆ?\\n### ಉತ್ತರ:',\n",
    "]\n",
    "\n",
    "for prompt in test_prompts:\n",
    "    result = pipe(prompt)\n",
    "    generated = result[0]['generated_text']\n",
    "    answer = generated.split('### ಉತ್ತರ:')[-1].strip()\n",
    "    question = prompt.split('\\n')[1]\n",
    "    print(f'Q: {question}')\n",
    "    print(f'A: {answer[:200]}')\n",
    "    print('-' * 50)"
   ]
  }
 ]
}